# Step 6: 选型决策 — W4A16（访存墙）/ W8A8（计算墙）/ FP8

**目标**：建立量化方法的选型决策框架。用纯算术成本模型，根据 batch / 显存 / 吞吐目标，判断该选 W4A16（AWQ/GPTQ）、W8A8（SmoothQuant）、还是 FP8。**纯 CPU 概念 step，无 GPU 代码**。

**对应 OUTLINE 课时**：1.6 选型决策（~40 分钟）。

> 本 step 无 GPU 代码（NOTEBOOK_CONVENTIONS 例外：概念/选型 step 的 L3 是 CPU 成本模型 cell，无 `torch.cuda.is_available()` 守卫）。

In [ ]:
%%capture
import math, json, pathlib
import torch
import torch.nn as nn
import ipytest
ipytest.autoconfig()

In [ ]:
# Setup cell：notebook 向上发现模块根（含 steps/ + pyproject.toml），绝不依赖裸相对路径。
# 规范见 course/NOTEBOOK_CONVENTIONS.md 第 2 节。所有文件路径从 MODULE_ROOT 派生。
def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "steps").is_dir() and (cand / "pyproject.toml").exists():
            return cand
    raise RuntimeError("找不到模块根（含 steps/ + pyproject.toml）；请在模块目录内 cd course/m1-activation-outliers 启动 jupyter")

MODULE_ROOT    = _find_module_root(pathlib.Path.cwd())
MODEL_DIR      = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 scripts/download_model.sh 一致
TINY_MODEL_DIR = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"     # L3 先在 0.5B 上验，再上 7B
OUT_ROOT       = MODULE_ROOT / "out"                                 # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("MODULE_ROOT:", MODULE_ROOT)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(), "（sm{}{}，cap={}）".format(cap[0], cap[1], cap),
          "— 支持 FP8" if cap >= (8, 9) else "")
else:
    print("无 GPU（仅 L1/L2 可跑；L3 自动跳过）")

## 原理：两堵墙

LLM 推理延迟受两种瓶颈限制，量化方法的选择本质是"你在撞哪堵墙"：

### 访存墙（Memory-bound）

当 batch 小、sequence 短时（典型对话场景 batch=1-8），每生成一个 token 只能复用一次权重——**搬权重的带宽**是瓶颈，算力大量闲置。
- **指标**：权重的字节数（bytes/token 决定延迟）。
- **对策**：**weight-only 量化**（W4A16，如 AWQ/GPTQ）——把权重压到 INT4（1/4 字节），激活仍 FP16。算力不变，但搬的字节少 4×，延迟近 4× 降。

### 计算墙（Compute-bound）

当 batch 大、prefill 长（如批量推理、长文档），每个权重被多次复用，算力被打满——**FLOPS** 成瓶颈。
- **指标**：需要硬件低精度算术单元（INT8/FP8 tensor core）。
- **对策**：**权重+激活都量化**（W8A8，如 SmoothQuant；或 FP8）——激活也变 INT8/FP8，能用低精度 tensor core 算 2-4× 快。

### 方法对照

| 方法 | 权重 | 激活 | 命中墙 | 硬件要求 |
|------|------|------|--------|----------|
| AWQ/GPTQ (W4A16) | INT4 | FP16 | 访存墙 | 任意（weight-only，通用） |
| SmoothQuant (W8A8) | INT8 | INT8 | 计算墙 | INT8 tensor core（A100+） |
| FP8 (W8A8 float) | E4M3 | E4M3 | 计算墙 | FP8 硬件（Hopper/Ada sm89+） |

**权重字节**：每参数位数 / 8 × 参数数。7B 模型：FP16=14GB，INT4=3.5GB，INT8/FP8=7GB。

### 选型规则（经验）

- **batch ≤ 8（对话/单用户）**：撞访存墙 → **W4A16（AWQ/GPTQ）**。最小字节，延迟最低。
- **batch 大（批量服务）+ 有 INT8 tensor core**：撞计算墙 → **W8A8（SmoothQuant）** 或 FP8。
- **有 FP8 硬件（H200/H100/L20X）**：**FP8 首选**——精度最高、无需校准、抗离群（s5）。
- **显存极度紧张**：W4A16（3.5GB）唯一选择（W8A8/FP8 都 7GB）。

## 本步填空

1. **`weight_bytes(n_params, method)`** —— 按方法算 7B 等模型的权重字节数。
2. **`recommend_method(batch_size, has_int8_tc, has_fp8_hw, memory_constrained)`** —— **判断型**：按约束推荐方法。

In [ ]:
def weight_bytes(n_params, method="fp16"):
    """按量化方法算模型权重的总字节数。

    参数
    ----
    n_params : int 或 float，模型参数数（如 7e9 = 7B）。
    method : str，之一：
      - "fp16"  : 每参数 16 bit = 2 byte
      - "int8"/"fp8" : 每参数 8 bit = 1 byte
      - "int4"/"w4a16" : 每参数 4 bit = 0.5 byte（weight-only，激活仍 FP16 不计入权重字节）

    返回
    ----
    int：权重总字节数。

    提示
    ----
      - bits_per_param = {"fp16":16, "int8":8, "fp8":8, "int4":4, "w4a16":4}[method]
      - bytes = n_params * bits / 8。
      - 返回 int（floor 即可）。
    """
    # TODO: 返回权重字节数。
    raise NotImplementedError


# 各方法的元数据（提供）
METHOD_META = {
    "w4a16":   {"name": "AWQ/GPTQ",  "w_bits": 4,  "a_bits": 16, "hits": "访存墙", "hw": "任意（weight-only）"},
    "w8a8":    {"name": "SmoothQuant","w_bits": 8,  "a_bits": 8,  "hits": "计算墙", "hw": "INT8 tensor core (A100+)"},
    "fp8":     {"name": "FP8 E4M3",   "w_bits": 8,  "a_bits": 8,  "hits": "计算墙", "hw": "FP8 硬件 (sm89+)"},
    "fp16":    {"name": "FP16 基线",  "w_bits": 16, "a_bits": 16, "hits": "—",      "hw": "任意"},
}

In [ ]:
def recommend_method(batch_size, has_int8_tc=False, has_fp8_hw=False, memory_constrained=False):
    """按部署约束推荐量化方法（判断型核心）。

    参数
    ----
    batch_size : int，并发 batch（决定撞哪堵墙）。
    has_int8_tc : bool，是否有 INT8 tensor core（A100+）。
    has_fp8_hw : bool，是否有 FP8 硬件（sm89+：H100/H200/L20X/L40S）。
    memory_constrained : bool，是否显存极度紧张（必须最小字节）。

    返回
    ----
    str：推荐的方法 key（"w4a16" / "w8a8" / "fp8"）。

    决策规则（按优先级，对齐原理 cell）：
      1. 显存极度紧张 → "w4a16"（3.5GB，唯一放得下）。
      2. 有 FP8 硬件 → "fp8"（精度最高、抗离群、无需校准——首选）。
      3. batch <= 8（对话，撞访存墙）→ "w4a16"（最小字节）。
      4. batch > 8 且有 INT8 tensor core → "w8a8"（计算墙，低精度算）。
      5. 其余（batch 大但无 INT8/FP8 硬件）→ "w4a16"（退化为 weight-only）。

    提示（判断训练点）：
      - 优先级顺序很重要：显存 > FP8 硬件 > batch 判墙 > INT8 硬件。
        想想为什么 FP8 硬件优先级高于 batch 判墙——因为 FP8 同时解决精度、抗离群、
        且在两堵墙上都不亏（8 bit 权重 + tensor core 算力）。
      - 别只返回字面量，要按规则走完判断链。
    """
    # TODO: 实现决策规则，返回方法 key。
    raise NotImplementedError

In [ ]:
%%ipytest -qq

def test_weight_bytes_values():
    assert weight_bytes(7e9, "fp16")  == int(7e9 * 16 / 8)   # 14e9
    assert weight_bytes(7e9, "int8")  == int(7e9 * 8 / 8)    # 7e9
    assert weight_bytes(7e9, "fp8")   == int(7e9 * 8 / 8)
    assert weight_bytes(7e9, "int4")  == int(7e9 * 4 / 8)    # 3.5e9
    assert weight_bytes(7e9, "w4a16") == int(7e9 * 4 / 8)

def test_recommend_memory_constrained_wins():
    # 显存紧张优先，即使有 FP8 硬件
    assert recommend_method(batch_size=64, has_fp8_hw=True, memory_constrained=True) == "w4a16"

def test_recommend_fp8_when_hw_available():
    assert recommend_method(batch_size=64, has_int8_tc=True, has_fp8_hw=True) == "fp8"

def test_recommend_w4a16_for_small_batch_no_hw():
    # 对话场景 batch=1，无特殊硬件 → 访存墙 → w4a16
    assert recommend_method(batch_size=1) == "w4a16"

def test_recommend_w4a16_for_conversational_batch():
    assert recommend_method(batch_size=8, has_int8_tc=True) == "w4a16"

def test_recommend_w8a8_large_batch_with_int8():
    # batch 大 + INT8 tensor core，无 FP8 → 计算墙 → w8a8
    assert recommend_method(batch_size=64, has_int8_tc=True, has_fp8_hw=False) == "w8a8"

def test_recommend_falls_back_to_w4a16_without_hw():
    # batch 大但无任何低精度硬件 → 退化 weight-only
    assert recommend_method(batch_size=64, has_int8_tc=False, has_fp8_hw=False) == "w4a16"

## L2：tiny 成本模型交叉点（CPU）

纯算术：在 H200（假设带宽/算力参数）上，画 W4A16 vs W8A8/FP8 的延迟 vs batch 曲线，找交叉点（batch 多大时从访存墙切到计算墙）。

In [ ]:
# H200 典型参数（近似，用于教学成本模型，非精确基准）
H200_MEM_BW = 4.8e12      # 4.8 TB/s 显存带宽
H200_INT8_TFLOPS = 1979e12   # INT8 tensor core ~1979 TFLOPS
H200_FP8_TFLOPS  = 3958e12   # FP8 tensor core ~3958 TFLOPS
H200_FP16_TFLOPS = 989e12    # FP16 ~989 TFLOPS

def decode_latency_per_token(n_params, method, batch_size, mem_bw, flops):
    """简化每 token 解码延迟（取访存与计算两者的 max）。"""
    w_bytes = weight_bytes(n_params, method)
    mem_time = w_bytes / mem_bw / batch_size   # batch 内复用权重，除 batch
    # 每个 token 每 params 约 2 FLOP（矩阵向量）；激活低精度时用对应 flops
    a_method = method if method != "w4a16" else "fp16"   # w4a16 激活 FP16
    tflops = {"fp16":H200_FP16_TFLOPS, "int8":H200_INT8_TFLOPS,
              "fp8":H200_FP8_TFLOPS, "w4a16":H200_FP16_TFLOPS}[a_method]
    compute_time = (2 * n_params) / tflops
    return max(mem_time, compute_time)

n_params = 7e9
batches = [1, 2, 4, 8, 16, 32, 64, 128]
print("batch | W4A16(ms) | W8A8(ms) | FP8(ms) | 瓶颈切换")
prev_winner = None
for b in batches:
    t4 = decode_latency_per_token(n_params, "w4a16", b, H200_MEM_BW, None) * 1e3
    t8 = decode_latency_per_token(n_params, "int8",  b, H200_MEM_BW, None) * 1e3
    tf = decode_latency_per_token(n_params, "fp8",   b, H200_MEM_BW, None) * 1e3
    winner = min([("W4A16",t4),("W8A8",t8),("FP8",tf)], key=lambda x:x[1])[0]
    switch = "  <-- 切到 " + winner if winner != prev_winner else ""
    print(f"{b:5d} | {t4:9.3f} | {t8:8.3f} | {tf:7.3f} | {winner}{switch}")
    prev_winner = winner
print("\n观察：batch 小时 W4A16 赢（访存墙，字节少）；batch 大时 FP8/W8A8 赢（计算墙，低精度算）。")
print("L2 PASS：成本模型交叉点计算跑通")

## L3：选型表（CPU 概念，无 GPU）

按若干典型部署场景跑 `recommend_method`，生成 `selection_table.json`。**无 GPU 代码**（概念 step 例外）。

In [ ]:
scenarios = [
    {"name": "单用户对话 (batch=1, 无特殊硬件)", "batch_size":1, "has_int8_tc":False, "has_fp8_hw":False},
    {"name": "对话服务 (batch=8, A100)", "batch_size":8, "has_int8_tc":True, "has_fp8_hw":False},
    {"name": "批量推理 (batch=64, A100)", "batch_size":64, "has_int8_tc":True, "has_fp8_hw":False},
    {"name": "H200 服务 (batch=64, FP8)", "batch_size":64, "has_int8_tc":True, "has_fp8_hw":True},
    {"name": "边缘部署 (显存紧张)", "batch_size":1, "memory_constrained":True, "has_fp8_hw":True},
    {"name": "大 batch 无低精度硬件", "batch_size":128, "has_int8_tc":False, "has_fp8_hw":False},
]
table = []
for s in scenarios:
    rec = recommend_method(**{k:v for k,v in s.items() if k!="name"})
    w_bytes_7b = weight_bytes(7e9, rec if rec!="w8a8" else "int8")
    table.append({"scenario": s["name"], "recommended": rec,
                  "method_name": METHOD_META[rec]["name"],
                  "7B_weight_GB": round(w_bytes_7b/1e9, 2),
                  "hits": METHOD_META[rec]["hits"]})
    print(f"{s['name']:42s} -> {METHOD_META[rec]['name']:12s} ({rec}), "
          f"7B 权重 {w_bytes_7b/1e9:.1f}GB, 命中 {METHOD_META[rec]['hits']}")

(OUT_ROOT/"selection_table.json").write_text(json.dumps(table, indent=2, ensure_ascii=False))
print("\nL3 PASS：选型表写入", OUT_ROOT/"selection_table.json")

## 产物检查

打印 `selection_table.json`，确认决策与原理一致。

In [ ]:
def report_selection():
    path = OUT_ROOT / "selection_table.json"
    table = json.loads(path.read_text())
    print("=== 量化方法选型表 ===")
    print(f"{'场景':42s} {'推荐方法':16s} {'7B权重GB':10s} {'瓶颈'}")
    for r in table:
        print(f"{r['scenario']:42s} {r['method_name']:16s} {r['7B_weight_GB']:<10} {r['hits']}")
    print("\n决策一致性检查：")
    # 抽查：显存紧张必 w4a16
    edge = [r for r in table if "显存" in r["scenario"]][0]
    assert edge["recommended"] == "w4a16", "显存紧张必须 w4a16"
    h200 = [r for r in table if "H200" in r["scenario"]][0]
    assert h200["recommended"] == "fp8", "H200+FP8 硬件必须 fp8"
    print("  ✓ 显存紧张→w4a16，H200+FP8→fp8，决策与原理一致")

report_selection()